##End-to-end MAG reconstruction

STEP 1. Genome Assembly

In [7]:
!wget -O reads.qza \
    https://polybox.ethz.ch/index.php/s/pNZbKcaKrwSiFMH/download


--2026-06-22 16:47:14--  https://polybox.ethz.ch/index.php/s/pNZbKcaKrwSiFMH/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 548786775 (523M) [application/octet-stream]
Saving to: ‘reads.qza’

reads.qza           100%[===================>] 523.36M  1.03MB/s    in 10m 40s 

2026-06-22 16:57:56 (837 KB/s) - ‘reads.qza’ saved [548786775/548786775]



In [8]:
%%writefile parallel.config.toml
[parsl]

[[parsl.executors]]
class = "HighThroughputExecutor"
label = "default"

[parsl.executors.provider]
class = "LocalProvider"
max_blocks = 2

Overwriting parallel.config.toml


Sample 2 and sample 3 analysis

In [9]:
import subprocess
import os
import time

# Verify QIIME and environment
result = subprocess.run(['qiime', '--version'], capture_output=True, text=True)
print('QIIME:', result.stdout.strip())

# Check working directory
print('Directory:', os.getcwd())

# Check available files
print('\nAvailable files:')
for f in ['reads.qza', 'reference-genomes.qza', 'busco-db-bacteria.qza']:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'  {status} {f}')

QIIME: q2cli version 2026.4.0
Run `qiime info` for more version details.
Directory: /home/abbasi/Documents/metagenomics

Available files:
  ✅ reads.qza
  ✅ reference-genomes.qza
  ✅ busco-db-bacteria.qza


In [10]:
# Create metadata for sample2 + sample3
with open('samples_2_3.tsv', 'w') as f:
    f.write('#SampleID\n')
    f.write('sample2\n')
    f.write('sample3\n')

print('Created samples_2_3.tsv:')
with open('samples_2_3.tsv') as f:
    print(f.read())

Created samples_2_3.tsv:
#SampleID
sample2
sample3



In [11]:
cmd = [
    'qiime', 'demux', 'filter-samples',
    '--i-demux', 'reads.qza',
    '--m-metadata-file', 'samples_2_3.tsv',
    '--o-filtered-demux', 'reads_23.qza'
]

print('Filtering reads...')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if os.path.exists('reads_23.qza'):
    size_mb = os.path.getsize('reads_23.qza') / 1024 / 1024
    print(f'\n✅ reads_23.qza created ({size_mb:.1f} MB)')
else:
    print('\n❌ Failed')

Filtering reads...
Saved SampleData[PairedEndSequencesWithQuality] to: reads_23.qza


✅ reads_23.qza created (184.7 MB)


Genome assembly

In [12]:
cmd = [
    'qiime', 'assembly', 'assemble-megahit',
    '--i-reads', 'reads_23.qza',
    '--p-presets', 'meta-sensitive',
    '--p-num-cpu-threads', '2',
    '--p-memory', '0.6',
    '--p-min-contig-len', '500',
    '--o-contigs', 'contigs-23.qza',
    '--verbose'
]

print('=' * 50)
print('Starting MEGAHIT assembly...')
print('=' * 50)

start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
elapsed = (time.time() - start) / 60

print(f'\nDone in {elapsed:.1f} minutes')

if os.path.exists('contigs-23.qza'):
    size_mb = os.path.getsize('contigs-23.qza') / 1024 / 1024
    print(f'✅ contigs-23.qza ({size_mb:.1f} MB)')
else:
    print('❌ Assembly failed')

Starting MEGAHIT assembly...


Quality control

In [ ]:
cmd = [
    'qiime', 'assembly', 'evaluate-quast',
    '--i-contigs', 'contigs-23.qza',
    '--i-references', 'reference-genomes.qza',
    '--p-threads', '2',
    '--p-min-contig', '500',
    '--o-reference-genomes', 'quast-ref-genomes-23.qza',
    '--o-results-table', 'quast-results-table-23.qza',
    '--o-visualization', 'quast-results-23.qzv',
    '--verbose'
]

print('Running QUAST with reference genomes...')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

outputs = {
    'quast-results-table-23.qza': 'QUAST results table',
    'quast-results-23.qzv': 'QUAST visualization',
    'quast-ref-genomes-23.qza': 'Reference genomes'
}

print('\n' + '=' * 50)
for filename, description in outputs.items():
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / 1024 / 1024
        print(f'✅ {filename:<35} {size_mb:>8.1f} MB | {description}')
    else:
        print(f'❌ {filename:<35} {"MISSING":>10} | {description}')

Running QUAST with reference genomes...
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metaquast.py -o /tmp/tmpy9_v8a0_/results --min-contig 500 --threads 2 --k-mer-size 101 --contig-thresholds 0,1000,5000,10000,25000,50000 --min-alignment 65 --min-identity 90.0 --ambiguity-usage one --ambiguity-score 0.99 /tmp/qiime2/abbasi/data/8136cf02-5d27-4f69-99ee-7442055ca086/data/sample2_contigs.fa /tmp/qiime2/abbasi/data/8136cf02-5d27-4f69-99ee-7442055ca086/data/sample3_contigs.fa -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/B_subtilis.fasta -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/E_coli_K-12.fasta -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/E_coli_O157_H7.fasta -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb

MAG Binning
i.Contig indexing
ii.Read mapping
iii. Contig binning
iv. MAG quality control 

In [13]:
import subprocess
import os
import time

cmd = [
    'qiime', 'assembly', 'index-contigs',
    '--i-contigs', 'contigs-23.qza',
    '--o-index', 'contigs-index-23.qza',
    '--verbose'
]

print('=' * 50)
print('Indexing contigs for sample2 + sample3...')
print('=' * 50)

start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
elapsed = (time.time() - start) / 60

print(f'\nDone in {elapsed:.1f} minutes')

if os.path.exists('contigs-index-23.qza'):
    size_mb = os.path.getsize('contigs-index-23.qza') / 1024 / 1024
    print(f'✅ contigs-index-23.qza created ({size_mb:.1f} MB)')
else:
    print('❌ Indexing failed')


Indexing contigs for sample2 + sample3...
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bowtie2-build --bmaxdivn 4 --dcv 1024 --offrate 5 --ftabchars 10 --threads 1 /tmp/qiime2/abbasi/data/340a9638-25ef-4cf6-8f47-5093b60a0046/data/sample2_contigs.fa /tmp/qiime2/abbasi/processes/125938-1782121771.65@abbasi/tmp/rachis-OutPath-hvg0jyzw/sample2/index

Settings:
  Output files: "/tmp/qiime2/abbasi/processes/125938-1782121771.65@abbasi/tmp/rachis-OutPath-hvg0jyzw/sample2/index.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 5 (one in 32)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local

In [14]:
cmd = [
    'qiime', 'assembly', 'map-reads',
    '--i-index', 'contigs-index-23.qza',
    '--i-reads', 'reads_23.qza',
    '--p-threads', '2',
    '--p-seed', '100',
    '--o-alignment-maps', 'reads-to-contigs-aln-23.qza',
    '--verbose'
]

print('=' * 50)
print('Starting map-reads...')
print('=' * 50)

start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
elapsed = (time.time() - start) / 60

print(f'\nDone in {elapsed:.1f} minutes')

if os.path.exists('reads-to-contigs-aln-23.qza'):
    size_mb = os.path.getsize('reads-to-contigs-aln-23.qza') / 1024 / 1024
    print(f'✅ reads-to-contigs-aln-23.qza ({size_mb:.1f} MB)')
else:
    print('❌ Failed')

Starting map-reads...
/home/abbasi/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bowtie2 -L 22 -i S,1,1.15 --n-ceil L,0,0.15 --dpad 15 --gbar 4 --ma 2 --mp 6 --np 1 --rdg 5,3 --rfg 5,3 -D 15 -R 2 --maxins 500 --fr --threads 2 --seed 100 --sensitive-local -x /tmp/qiime2/abbasi/data/401122c9-3fc0-4c05-a26e-12197ec468db/data/sample2/index -1 /tmp/qiime2/abbasi/data/c61f790d-4112-45e4-bd02-b4e980a252fb/data/sample2_00_L001_R1_001.fastq.gz -2 /tmp/qiime2/abbasi/data/c61f790d-4112-45e4-bd02-b4e980a252fb/data/sample2_00_L001_R2_001.fastq.gz | 

In [18]:
cmd = [
    'qiime', 'annotate', 'bin-contigs-metabat',
    '--i-contigs', 'contigs-23.qza',
    '--i-alignment-maps', 'reads-to-contigs-aln-23.qza',
    '--p-num-threads', '2',
    '--p-seed', '100',
    '--o-mags', 'mags-23.qza',
    '--o-contig-map', 'contig-map-23.qza',
    '--o-unbinned-contigs', 'unbinned-contigs-23.qza',
    '--verbose'
]

print('=' * 50)
print('Starting MAG binning with MetaBAT 2...')
print('Plugin: annotate | Tool: MetaBAT 2')
print('=' * 50)

start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
elapsed = (time.time() - start) / 60

print(f'\nDone in {elapsed:.1f} minutes')

outputs = {
    'mags-23.qza': 'MAGs (binned genomes)',
    'contig-map-23.qza': 'Contig-to-MAG mapping',
    'unbinned-contigs-23.qza': 'Unbinned contigs'
}

for filename, description in outputs.items():
    if os.path.exists(filename):
        size_mb = os.path.getsize(filename) / 1024 / 1024
        print(f'✅ {filename:<35} {size_mb:>8.1f} MB | {description}')
    else:
        print(f'❌ {filename:<35} {"MISSING":>10} | {description}')

Starting MAG binning with MetaBAT 2...
Plugin: annotate | Tool: MetaBAT 2
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools sort /tmp/qiime2/abbasi/data/a2c2a895-fe6e-4f16-ba9f-88c099f6cca1/data/sample2_alignment.bam -o /tmp/tmpcpgshyre/sample2_alignment_sorted.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: jgi_summarize_bam_contig_depths --outputDepth /tmp/tmpcpgshyre/sample2_depth.txt /tmp/tmpcpgshyre/sample2_alignment_sorted.bam

Output depth matrix to /tmp/tmpcpgshyre/sample2_depth.txt
jgi_summarize_bam_contig_depths 2.18 20250528_211447
Running with 8 threads to save memory you can red

In [10]:
import subprocess

def sort_bam(input_bam, output_bam):
    cmd = ["samtools", "sort", input_bam, "-o", output_bam]
    print(f"Sorting: {input_bam} → {output_bam}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        return False
    print("✅ Sorting complete!")
    return True

# Sort both BAMs
sort_bam(
    "/home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.bam",
    "/home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.sorted.bam"
)

sort_bam(
    "/home/abbasi/Documents/metagenomics/exported_bam/sample3_alignment.bam",
    "/home/abbasi/Documents/metagenomics/exported_bam/sample3_alignment.sorted.bam"
)

Sorting: /home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.bam → /home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.sorted.bam
✅ Sorting complete!
Sorting: /home/abbasi/Documents/metagenomics/exported_bam/sample3_alignment.bam → /home/abbasi/Documents/metagenomics/exported_bam/sample3_alignment.sorted.bam
✅ Sorting complete!


True

In [16]:
# ============================================
# MetaBAT2 Binning for Samples 2 & 3
# ============================================

import subprocess
import os

# --- CONFIGURATION ---
SAMPLE2_CONTIGS = "/home/abbasi/Documents/metagenomics/exported_contigs/sample2_contigs.fa"
SAMPLE3_CONTIGS = "/home/abbasi/Documents/metagenomics/exported_contigs/sample3_contigs.fa"

# ✅ USE SORTED BAM FILES
SAMPLE2_BAM = "/home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.sorted.bam"
SAMPLE3_BAM = "/home/abbasi/Documents/metagenomics/exported_bam/sample3_alignment.sorted.bam"

OUTPUT_DIR = "/home/abbasi/metabat2_bins"
MIN_CONTIG_SIZE = 1500
MIN_BIN_SIZE = 200000
THREADS = 4

# Create output directories
os.makedirs(f"{OUTPUT_DIR}/sample2", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/sample3", exist_ok=True)

# ============================================
# FUNCTION: Generate depth file from BAM
# ============================================
def generate_depth_file(contigs_fasta, bam_file, output_depth):
    cmd = [
        "jgi_summarize_bam_contig_depths",
        "--outputDepth", output_depth,
        "--referenceFasta", contigs_fasta,
        bam_file
    ]
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
    return result.returncode == 0

# ============================================
# FUNCTION: Run MetaBAT2
# ============================================
def run_metabat2(contigs_fasta, depth_file, output_prefix, 
                 min_contig=MIN_CONTIG_SIZE, 
                 min_bin_size=MIN_BIN_SIZE, 
                 threads=THREADS):
    cmd = [
        "metabat2",
        "-i", contigs_fasta,
        "-a", depth_file,
        "-o", output_prefix,
        "-m", str(min_contig),
        "-s", str(min_bin_size),
        "-t", str(threads),
        "--unbinned",
        "--verbose"
    ]
    print(f"\nRunning: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
    return result.returncode == 0

# ============================================
# RUN BINNING FOR SAMPLE 2
# ============================================
print("=" * 50)
print("PROCESSING SAMPLE 2")
print("=" * 50)

sample2_depth = f"{OUTPUT_DIR}/sample2/sample2_depth.txt"
sample2_bins = f"{OUTPUT_DIR}/sample2/sample2_bin"

success = generate_depth_file(SAMPLE2_CONTIGS, SAMPLE2_BAM, sample2_depth)
if success:
    run_metabat2(SAMPLE2_CONTIGS, sample2_depth, sample2_bins)
    print(f"\n✅ Sample 2 bins saved to: {OUTPUT_DIR}/sample2/")
else:
    print("❌ Failed to generate depth file for Sample 2")

# ============================================
# RUN BINNING FOR SAMPLE 3
# ============================================
print("\n" + "=" * 50)
print("PROCESSING SAMPLE 3")
print("=" * 50)

sample3_depth = f"{OUTPUT_DIR}/sample3/sample3_depth.txt"
sample3_bins = f"{OUTPUT_DIR}/sample3/sample3_bin"

success = generate_depth_file(SAMPLE3_CONTIGS, SAMPLE3_BAM, sample3_depth)
if success:
    run_metabat2(SAMPLE3_CONTIGS, sample3_depth, sample3_bins)
    print(f"\n✅ Sample 3 bins saved to: {OUTPUT_DIR}/sample3/")
else:
    print("❌ Failed to generate depth file for Sample 3")

# ============================================
# CHECK RESULTS
# ============================================
print("\n" + "=" * 50)
print("BINNING RESULTS")
print("=" * 50)

for sample in ["sample2", "sample3"]:
    sample_dir = f"{OUTPUT_DIR}/{sample}"
    if os.path.exists(sample_dir):
        all_files = [f for f in os.listdir(sample_dir) if f.endswith(".fa")]
        bins = [f for f in all_files if "bin" in f and "unbinned" not in f]
        unbinned = [f for f in all_files if "unbinned" in f]
        print(f"\n{sample.upper()}:")
        print(f"  - Total .fa files: {len(all_files)}")
        print(f"  - Bins: {len(bins)}")
        print(f"  - Unbinned: {len(unbinned)}")
        for b in sorted(bins)[:5]:
            filepath = os.path.join(sample_dir, b)
            size = os.path.getsize(filepath)
            print(f"    {b} ({size:,} bytes)")
        if len(bins) > 5:
            print(f"    ... and {len(bins)-5} more bins")
    else:
        print(f"\n{sample.upper()}: Directory not found")

PROCESSING SAMPLE 2
Running: jgi_summarize_bam_contig_depths --outputDepth /home/abbasi/metabat2_bins/sample2/sample2_depth.txt --referenceFasta /home/abbasi/Documents/metagenomics/exported_contigs/sample2_contigs.fa /home/abbasi/Documents/metagenomics/exported_bam/sample2_alignment.sorted.bam
Reading reference fasta file: /home/abbasi/Documents/metagenomics/exported_contigs/sample2_contigs.fa
... 10200 sequences


Running: metabat2 -i /home/abbasi/Documents/metagenomics/exported_contigs/sample2_contigs.fa -a /home/abbasi/metabat2_bins/sample2/sample2_depth.txt -o /home/abbasi/metabat2_bins/sample2/sample2_bin -m 1500 -s 200000 -t 4 --unbinned --verbose
MetaBAT 2 (2.18) using minContig 1500, minCV 1.0, minCVSum 1.0, maxP 95%, minS 60, maxEdges 200 and minClsSize 200000. with random seed=1782131552
[00:00:00] Executing with 4 threads
[00:00:00] Parsing abundance file header [1.1Gb / 4.8Gb]
[00:00:00] Parsing assembly file [1.1Gb / 4.8Gb]
[00:00:00] Outputting contigs that are too short 

In [1]:
# ============================================
# MAG Quality Estimation from MetaBAT2 Bins
# ============================================

import os

METABAT2_DIR = "/home/abbasi/metabat2_bins"

# ============================================
# FUNCTION: Calculate bin statistics
# ============================================
def analyze_bin(fasta_path):
    """Calculate basic stats for a bin FASTA file."""
    sequences = []
    current_seq = []
    
    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_seq:
                    sequences.append(''.join(current_seq))
                    current_seq = []
            else:
                current_seq.append(line)
        if current_seq:
            sequences.append(''.join(current_seq))
    
    lengths = [len(seq) for seq in sequences]
    total_bp = sum(lengths)
    n_contigs = len(sequences)
    
    # GC content
    gc_count = sum(seq.count('G') + seq.count('C') for seq in sequences)
    at_count = sum(seq.count('A') + seq.count('T') for seq in sequences)
    gc_content = (gc_count / (gc_count + at_count)) * 100 if (gc_count + at_count) > 0 else 0
    
    # N50
    sorted_lengths = sorted(lengths, reverse=True)
    cumulative = 0
    n50 = 0
    for l in sorted_lengths:
        cumulative += l
        if cumulative >= total_bp / 2:
            n50 = l
            break
    
    return {
        'file': os.path.basename(fasta_path),
        'total_bp': total_bp,
        'n_contigs': n_contigs,
        'n50': n50,
        'gc_content': gc_content,
        'max_contig': max(lengths) if lengths else 0,
        'min_contig': min(lengths) if lengths else 0
    }

# ============================================
# Analyze all bins
# ============================================
print("=" * 70)
print("META BAT2 BIN QUALITY SUMMARY")
print("=" * 70)

for sample in ["sample2", "sample3"]:
    print(f"\n{'='*70}")
    print(f"  {sample.upper()}")
    print(f"{'='*70}")
    
    sample_dir = f"{METABAT2_DIR}/{sample}"
    bins = [f for f in os.listdir(sample_dir) 
            if f.endswith(".fa") and "bin." in f 
            and "unbinned" not in f and "tooShort" not in f and "lowDepth" not in f]
    
    results = []
    for bin_file in sorted(bins):
        bin_path = f"{sample_dir}/{bin_file}"
        stats = analyze_bin(bin_path)
        results.append(stats)
    
    # Sort by total size (descending)
    results.sort(key=lambda x: x['total_bp'], reverse=True)
    
    # Print table
    print(f"\n{'Bin':<20} {'Size (Mb)':<12} {'Contigs':<10} {'N50':<12} {'GC%':<8} {'Assessment'}")
    print("-" * 70)
    
    for r in results:
        size_mb = r['total_bp'] / 1_000_000
        assessment = ""
        
        # Rough quality assessment based on size
        if size_mb >= 4.0:
            assessment = "⭐ High quality"
        elif size_mb >= 2.0:
            assessment = "✅ Medium quality"
        elif size_mb >= 1.0:
            assessment = "⚠️ Low quality"
        else:
            assessment = "❌ Too small"
        
        print(f"{r['file']:<20} {size_mb:<12.2f} {r['n_contigs']:<10} {r['n50']:<12,} {r['gc_content']:<8.1f} {assessment}")
    
    # Summary
    total_bins = len(results)
    high_quality = sum(1 for r in results if r['total_bp'] >= 4_000_000)
    medium_quality = sum(1 for r in results if 2_000_000 <= r['total_bp'] < 4_000_000)
    
    print(f"\n{'='*70}")
    print(f"  Summary: {total_bins} bins total")
    print(f"  - High quality (≥4 Mb): {high_quality}")
    print(f"  - Medium quality (2-4 Mb): {medium_quality}")
    print(f"  - Low quality (<2 Mb): {total_bins - high_quality - medium_quality}")
    print(f"{'='*70}")

META BAT2 BIN QUALITY SUMMARY

  SAMPLE2

Bin                  Size (Mb)    Contigs    N50          GC%      Assessment
----------------------------------------------------------------------
sample2_bin.1.fa     4.97         1199       4,969        50.4     ⭐ High quality
sample2_bin.5.fa     3.78         183        34,639       44.1     ✅ Medium quality
sample2_bin.3.fa     2.12         695        3,252        37.6     ✅ Medium quality
sample2_bin.4.fa     1.43         635        2,215        65.5     ⚠️ Low quality
sample2_bin.2.fa     1.05         427        2,471        54.1     ⚠️ Low quality
sample2_bin.6.fa     0.36         22         23,223       36.5     ❌ Too small

  Summary: 6 bins total
  - High quality (≥4 Mb): 1
  - Medium quality (2-4 Mb): 2
  - Low quality (<2 Mb): 3

  SAMPLE3

Bin                  Size (Mb)    Contigs    N50          GC%      Assessment
----------------------------------------------------------------------
sample3_bin.5.fa     4.84         126       

In [2]:
import subprocess
import os
import glob

METABAT2_DIR = "/home/abbasi/metabat2_bins"
BUSCO_OUT = "/home/abbasi/busco_all_results"
os.makedirs(BUSCO_OUT, exist_ok=True)

def run_busco(fasta, out_dir):
    cmd = ["busco", "-i", fasta, "-l", "bacteria_odb10", "-m", "genome", "-o", out_dir, "--force"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result

# Run on all bins
for sample in ["sample2", "sample3"]:
    sample_dir = f"{METABAT2_DIR}/{sample}"
    bins = glob.glob(f"{sample_dir}/*_bin.*.fa")
    for bin_path in sorted(bins):
        name = os.path.basename(bin_path).replace(".fa", "")
        out = f"{BUSCO_OUT}/{sample}_{name}"
        print(f"\nRunning BUSCO on {name}...")
        result = run_busco(bin_path, out)
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)  # Print last 500 chars



Running BUSCO on sample2_bin.1...
------------------------
2026-06-22 21:02:45 INFO:	BUSCO analysis done. Total running time: 19 seconds
2026-06-22 21:02:45 INFO:	Results written in /home/abbasi/Documents/metagenomics/home/abbasi/busco_all_results/sample2_sample2_bin.1
2026-06-22 21:02:45 INFO:	For assistance with interpreting the results, please consult the userguide: https://busco.ezlab.org/busco_userguide.html

2026-06-22 21:02:45 INFO:	Visit this page https://gitlab.com/ezlab/busco#how-to-cite-busco to see how to cite BUSCO


Running BUSCO on sample2_bin.2...
------------------------
2026-06-22 21:03:03 INFO:	BUSCO analysis done. Total running time: 10 seconds
2026-06-22 21:03:03 INFO:	Results written in /home/abbasi/Documents/metagenomics/home/abbasi/busco_all_results/sample2_sample2_bin.2
2026-06-22 21:03:03 INFO:	For assistance with interpreting the results, please consult the userguide: https://busco.ezlab.org/busco_userguide.html

2026-06-22 21:03:03 INFO:	Visit this page htt

In [8]:
# ============================================
# Parse All BUSCO Results
# ============================================

import os
import re

BUSCO_DIR = "/home/abbasi/busco_all_results"

# ============================================
# FUNCTION: Parse BUSCO short_summary.txt
# ============================================
def parse_busco_summary(summary_path):
    """Extract BUSCO metrics from short_summary.txt."""
    results = {}
    
    with open(summary_path, 'r') as f:
        content = f.read()
    
    # Extract the main line: C:82.3%[S:75.0%,D:7.3%],F:9.7%,M:8.0%,n:124
    match = re.search(r'C:(\d+\.?\d*)%\[S:(\d+\.?\d*)%,D:(\d+\.?\d*)%\],F:(\d+\.?\d*)%,M:(\d+\.?\d*)%,n:(\d+)', content)
    if match:
        results['complete'] = float(match.group(1))
        results['single'] = float(match.group(2))
        results['duplicated'] = float(match.group(3))
        results['fragmented'] = float(match.group(4))
        results['missing'] = float(match.group(5))
        results['total'] = int(match.group(6))
    
    # Extract individual counts
    complete_count = re.search(r'(\d+)\s+Complete BUSCOs \(C\)', content)
    single_count = re.search(r'(\d+)\s+Complete and single-copy BUSCOs \(S\)', content)
    dup_count = re.search(r'(\d+)\s+Complete and duplicated BUSCOs \(D\)', content)
    frag_count = re.search(r'(\d+)\s+Fragmented BUSCOs \(F\)', content)
    miss_count = re.search(r'(\d+)\s+Missing BUSCOs \(M\)', content)
    
    if complete_count: results['complete_count'] = int(complete_count.group(1))
    if single_count: results['single_count'] = int(single_count.group(1))
    if dup_count: results['dup_count'] = int(dup_count.group(1))
    if frag_count: results['frag_count'] = int(frag_count.group(1))
    if miss_count: results['miss_count'] = int(miss_count.group(1))
    
    return results

# ============================================
# Collect all results
# ============================================
print("=" * 80)
print("BUSCO RESULTS SUMMARY FOR ALL META BAT2 BINS")
print("=" * 80)

all_results = []

# Walk through busco output directories
for item in sorted(os.listdir(BUSCO_DIR)):
    item_path = os.path.join(BUSCO_DIR, item)
    if not os.path.isdir(item_path):
        continue
    
    # Find short_summary.txt
    summary_file = None
    for root, dirs, files in os.walk(item_path):
        for f in files:
            if f.startswith("short_summary") and f.endswith(".txt"):
                summary_file = os.path.join(root, f)
                break
        if summary_file:
            break
    
    if summary_file:
        busco_data = parse_busco_summary(summary_file)
        if busco_data:
            # Extract sample and bin name from directory name
            # e.g., "sample2_sample2_bin.1" -> sample2, bin.1
            parts = item.replace("sample2_sample2_", "sample2_").replace("sample3_sample3_", "sample3_").split("_")
            sample = parts[0] if len(parts) > 0 else "unknown"
            bin_name = "_".join(parts[1:]) if len(parts) > 1 else item
            
            all_results.append({
                'sample': sample,
                'bin': bin_name,
                **busco_data
            })

# ============================================
# Print summary table
# ============================================
# Sort by sample then by completeness
all_results.sort(key=lambda x: (x['sample'], -x.get('complete', 0)))

print(f"\n{'Sample':<10} {'Bin':<20} {'Complete%':<12} {'Single%':<10} {'Dup%':<8} {'Frag%':<8} {'Miss%':<8} {'Quality'}")
print("-" * 90)

for r in all_results:
    # Quality assessment
    comp = r.get('complete', 0)
    dup = r.get('duplicated', 0)
    
    if comp >= 90 and dup <= 5:
        quality = "⭐ Excellent"
    elif comp >= 70 and dup <= 10:
        quality = "✅ Good"
    elif comp >= 50:
        quality = "⚠️ Fair"
    elif comp > 0:
        quality = "❌ Poor"
    else:
        quality = "❌ No BUSCOs found"
    
    print(f"{r['sample']:<10} {r['bin']:<20} {comp:<12.1f} {r.get('single', 0):<10.1f} {dup:<8.1f} {r.get('fragmented', 0):<8.1f} {r.get('missing', 0):<8.1f} {quality}")

# ============================================
# Summary statistics
# ============================================
print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")

for sample in ["sample2", "sample3"]:
    sample_results = [r for r in all_results if r['sample'] == sample]
    if not sample_results:
        continue
    
    excellent = sum(1 for r in sample_results if r.get('complete', 0) >= 90 and r.get('duplicated', 0) <= 5)
    good = sum(1 for r in sample_results if r.get('complete', 0) >= 70 and r.get('duplicated', 0) <= 10)
    fair = sum(1 for r in sample_results if 50 <= r.get('complete', 0) < 70)
    poor = sum(1 for r in sample_results if 0 < r.get('complete', 0) < 50)
    no_busco = sum(1 for r in sample_results if r.get('complete', 0) == 0)
    
    print(f"\n{sample.upper()}:")
    print(f"  Total bins analyzed: {len(sample_results)}")
    print(f"  ⭐ Excellent (≥90% complete, ≤5% dup): {excellent}")
    print(f"  ✅ Good (≥70% complete, ≤10% dup): {good}")
    print(f"  ⚠️ Fair (50-70% complete): {fair}")
    print(f"  ❌ Poor (<50% complete): {poor}")
    print(f"  ❌ No BUSCOs found: {no_busco}")
    
    # Best bin
    best = max(sample_results, key=lambda x: x.get('complete', 0))
    print(f"  Best bin: {best['bin']} (C:{best.get('complete', 0):.1f}%, D:{best.get('duplicated', 0):.1f}%)")

BUSCO RESULTS SUMMARY FOR ALL META BAT2 BINS

Sample     Bin                  Complete%    Single%    Dup%     Frag%    Miss%    Quality
------------------------------------------------------------------------------------------

SUMMARY STATISTICS


In [ ]:
%%bash
qiime assembly assemble-megahit \
    --i-reads reads.qza \
    --p-presets meta-sensitive \
    --p-num-cpu-threads 2 \
    --p-min-contig-len 500 \
    --o-contigs contigs.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[Contigs] to: contigs.qza


In [ ]:
!wget -O reference-genomes.qza \
    https://polybox.ethz.ch/index.php/s/SDczx2T6sxnNL9x/download

--2026-06-20 10:03:27--  https://polybox.ethz.ch/index.php/s/SDczx2T6sxnNL9x/download
Resolving polybox.ethz.ch (polybox.ethz.ch)... 129.132.71.243
Connecting to polybox.ethz.ch (polybox.ethz.ch)|129.132.71.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11652707 (11M) [application/octet-stream]
Saving to: ‘reference-genomes.qza’

reference-genomes.q 100%[===================>]  11.11M   654KB/s    in 17s     

2026-06-20 10:03:46 (654 KB/s) - ‘reference-genomes.qza’ saved [11652707/11652707]



In [ ]:

%%bash
qiime assembly evaluate-quast \
    --i-contigs contigs.qza \
    --i-references reference-genomes.qza \
    --p-threads 4 \
    --p-min-contig 500 \
    --o-reference-genomes ref-genomes.qza \
    --o-results-table quast-results.qza \
    --o-visualization contigs-qc-quast.qzv \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: metaquast.py -o /tmp/tmpoqx1nfg_/results --min-contig 500 --threads 4 --k-mer-size 101 --contig-thresholds 0,1000,5000,10000,25000,50000 --min-alignment 65 --min-identity 90.0 --ambiguity-usage one --ambiguity-score 0.99 /tmp/qiime2/abbasi/data/ecf15a6c-e71b-47c7-aec6-105fe246120a/data/sample1_contigs.fa /tmp/qiime2/abbasi/data/ecf15a6c-e71b-47c7-aec6-105fe246120a/data/sample2_contigs.fa /tmp/qiime2/abbasi/data/ecf15a6c-e71b-47c7-aec6-105fe246120a/data/sample3_contigs.fa /tmp/qiime2/abbasi/data/ecf15a6c-e71b-47c7-aec6-105fe246120a/data/sample4_contigs.fa -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/B_subtilis.fasta -r /tmp/qiime2/abbasi/data/6545b7e5-a584-4bcc-bc41-484ed76555fb/data/E_coli_K-12.fasta -r /tmp/qiime2/abbasi/d

STEP 2. MAG Binning
i.Contig indexing
ii.Read mapping
iii. Contig binning
iv. MAG quality control

In [ ]:
%%bash
qiime assembly index-contigs \
    --i-contigs contigs.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-index contigs-index.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[SingleBowtie2Index % Properties('contigs')] to: contigs-index.qza


In [ ]:
%bash
qiime annotate bin-contigs-metabat \
    --i-contigs contigs.qza \
    --i-alignment-maps reads-to-contigs-aln.qza \
    --p-num-threads 4 \
    --p-seed 100 \
    --o-mags mags.qza \
    --o-contig-map contig-map.qza \
    --o-unbinned-contigs unbinned-contigs.qza \
    --verbose%


In [ ]:
%%bash
qiime annotate fetch-busco-db \
    --p-lineages bacteria_odb12 \
    --o-db busco-db-bacteria.qza \
    --verbose


Fetching lineages: bacteria_odb12.
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: busco --download_path /tmp/qiime2/sadia/processes/64914-1781030925.15@sadia/tmp/rachis-OutPath-6fyefww1 --download bacteria_odb12

2026-06-09 23:48:51 INFO:	Downloading information on latest versions of BUSCO data...
2026-06-09 23:49:00 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/lineages/bacteria_odb12.2026-05-22.tar.gz'
2026-06-09 23:52:35 INFO:	Decompressing file '/tmp/qiime2/sadia/processes/64914-1781030925.15@sadia/tmp/rachis-OutPath-6fyefww1/lineages/bacteria_odb12.tar.gz'
Download completed. 
Copying files from temporary directory to the final location...
Saved ReferenceDB[BUSCO] to: busco-db-bacteria.qza


In [ ]:
%%bash
qiime annotate evaluate-busco \
    --i-mags mags.qza \
    --i-db busco-db-bacteria.qza \
    --i-unbinned-contigs unbinned-contigs.qza \
    --p-lineage-dataset bacteria_odb12 \
    --p-cpu 2 \
    --o-results busco-results.qza \
    --o-visualization mags.qzv \
    --parallel-config parallel.config.toml \
    --verbose


Saved BUSCOResults to: busco-results.qza
Saved Visualization to: mags.qzv


In [ ]:
%%bash
qiime annotate filter-mags \
    --i-mags mags.qza \
    --m-metadata-file busco-results.qza \
    --p-where "completeness>50 AND contamination<10" \
    --p-on "mag" \
    --o-filtered-mags mags-filtered.qza \
    --verbose


/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing


Found 15 IDs to keep.
Saved SampleData[MAGs] to: mags-filtered.qza


STEP 3. MAG Dereplication

In [16]:
%%bash
qiime sourmash compute \
    --i-sequence-file mags-filtered.qza \
    --p-ksizes 105 \
    --p-scaled 100 \
    --o-min-hash-signature min-hash.qza \
    --verbose


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

** WARNING: the sourmash compute command is DEPRECATED as of 4.0 and
** will be removed in 5.0. Please see the 'sourmash sketch' command instead.

setting num_hashes to 0 because --scaled is set
computing signatures for files: /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/148b5aa1-e69f-4b20-b026-3874eb5545f9.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/3c8c0deb-fedb-4dea-95f6-d673f4bf7869.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/4b6e9c75-c335-4f4d-bd98-eba60255e450.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/5acc410b-8cef-434a-a75d-8e3b07b4da0e.fasta, /tmp/qiime2/sadia/processes/76749-1781031319.08@sadia/tmp/rachis-OutPath-vchjge_4/61af53fd-2310-40fe-ad72-9037ca2e240d.fasta, /tmp/qiime2/sadia/processes/76749-17810

Saved MinHashSig to: min-hash.qza


In [17]:
%%bash
qiime sourmash compare \
    --i-min-hash-signature min-hash.qza \
    --p-ksize 105 \
    --o-compare-output min-hash-compare.qza \
    --verbose


== This is sourmash version 4.9.4. ==
== Please cite Irber et. al (2024), doi:10.21105/joss.06830. ==

loaded 15 si                                                                   6b2-814a-44e3-9bb6-3874eb5545f9.fasta.sig'6-d673f4bf7869.fasta.sig'asta.sig'8-eba60255e450.fasta.sig'asta.sig'loading '/tmp/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/5acc410b-8cef-434a-a75d-8e3b07b4da0e.fasta.sig'.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/61af53fd-2310-40fe-ad72-9037ca2e240d.fasta.sig'5-ca88286c94b9.fasta.sig'asta.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/913f57a8-39a4-41ad-a742-27b01737baf8.fasta.sig'<<<-43b1-b432-dbb2d225f158/data/913f57a8-39a4-41ad-a742-27b01737baf8.fasta.sig'p/qiime2/sadia/data/75cf8ed6-1f75-43b1-b432-dbb2d225f158/data/9d6f671d-b2d7-423b-8fd1-a838dbc7666e.fasta.sig'4-8a2b664d1491.fasta.sig'asta.sig'8-0f16532c40f9.fasta.sig'asta.sig'2-dbb2d225f158/data/aeacdfca-ae18-4c48-8678-0f16532c40f9.fasta.sig'2-ecb3

0-/tmp/qiime2/sad...	[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
1-/tmp/qiime2/sad...	[0.    1.    0.    0.619 0.    0.    0.    0.    0.918 0.    0.    0.
 0.001 0.    0.006]
2-/tmp/qiime2/sad...	[0.    0.    1.    0.    0.    0.    0.    0.998 0.    0.    0.    0.
 0.    0.86  0.   ]
3-/tmp/qiime2/sad...	[0.    0.619 0.    1.    0.027 0.    0.    0.    0.621 0.    0.    0.
 0.    0.    0.027]
4-/tmp/qiime2/sad...	[0.    0.    0.    0.027 1.    0.    0.    0.    0.004 0.    0.    0.
 0.    0.    0.884]
5-/tmp/qiime2/sad...	[0.    0.    0.    0.    0.    1.    0.985 0.    0.    0.    0.    0.
 0.443 0.    0.   ]
6-/tmp/qiime2/sad...	[0.    0.    0.    0.    0.    0.985 1.    0.    0.    0.    0.    0.
 0.44  0.    0.   ]
7-/tmp/qiime2/sad...	[0.    0.    0.998 0.    0.    0.    0.    1.    0.    0.    0.    0.
 0.    0.861 0.   ]
8-/tmp/qiime2/sad...	[0.    0.918 0.    0.621 0.004 0.    0.    0.    1.    0.    0.    0.
 0.001 0.    0.   ]
9-/tmp/qiime2/sad...	[0.    0.    0.    0.  

STEP 4. Taxonomic classification
i.Database preparation
ii.Read-based classification
iii. MAG-based classification

In [18]:
%%bash
qiime annotate dereplicate-mags \
    --i-mags mags-filtered.qza \
    --i-distance-matrix min-hash-compare.qza \
    --m-metadata-file busco-results.qza \
    --p-metadata-column completeness \
    --p-threshold 0.9 \
    --o-dereplicated-mags mags-derep.qza \
    --o-table table.qza \
    --verbose

/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/rachis/metadata/metadata.py:610: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  series[missing.index] = missing
/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/scipy/cluster/hierarchy.py:810: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  return linkage(y, method='ward', metric='euclidean')


Saved FeatureData[MAG] to: mags-derep.qza
Saved FeatureTable[PresenceAbsence] to: table.qza


In [ ]:
%%bash
qiime annotate build-kraken-db \
    --p-collection standard8 \
    --o-kraken2-db kraken2-db.qza \
    --o-bracken-db bracken-db.qza \
    --verbose


Found the latest "standard8" database: kraken/k2_standard_08gb_20250402.tar.gz.
Download finished. Extracting database files... Done.
Saved Kraken2DB to: kraken2-db.qza
Saved BrackenDB to: bracken-db.qza


In [3]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs reads.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-reads.qza \
    --o-outputs kraken2-hits-reads.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved SampleData[Kraken2Report % Properties('reads')] to: kraken2-reports-reads.qza
Saved SampleData[Kraken2Output % Properties('reads')] to: kraken2-hits-reads.qza


In [4]:
%%bash
qiime annotate estimate-bracken \
    --i-kraken2-reports kraken2-reports-reads.qza \
    --i-db bracken-db.qza \
    --p-read-len 150 \
    --o-reports bracken-reports.qza \
    --o-taxonomy bracken-taxonomy.qza \
    --o-table bracken-table.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bracken -d /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1e-bf49-cc21e0082fd9/data -i /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt -o /tmp/tmp7zs48dur/sample3.bracken.output.txt -w /tmp/qiime2/sadia/processes/228314-1781094946.01@sadia/tmp/rachis-OutPath-luysp0hy/sample3.report.txt -t 0 -r 150 -l S

 >> Checking for Valid Options...
 >> Running Bracken 
      >> python src/est_abundance.py -i /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt -o /tmp/tmp7zs48dur/sample3.bracken.output.txt -k /tmp/qiime2/sadia/data/dd6f4be9-424c-4c1e-bf49-cc21e0082fd9/data/database150mers.kmer_distrib -l S -t 0
PROGRAM START TIME: 06-10-2026 12:36:03
BRACKEN SUMMARY (Kraken report: /tmp/qiime2/sadia/d

>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample3.report.txt
>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample2.report.txt
>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample4.report.txt
>> Checking report file: /tmp/qiime2/sadia/data/bed39717-48f4-4f41-b8df-c1f4d889b68c/data/sample1.report.txt
/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/q2_annotate/kraken2/select.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  table = pd.DataFrame(rows).fillna(False)


In [5]:
%%bash
qiime taxa barplot \
    --i-table bracken-table.qza \
    --i-taxonomy bracken-taxonomy.qza \
    --o-visualization bracken-barplot.qzv \
    --verbose


Saved Visualization to: bracken-barplot.qzv


In [6]:
%%bash
qiime annotate classify-kraken2 \
    --i-seqs mags-derep.qza \
    --i-db kraken2-db.qza \
    --p-threads 4 \
    --p-memory-mapping \
    --o-reports kraken2-reports-mags.qza \
    --o-outputs kraken2-hits-mags.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved FeatureData[Kraken2Report % Properties('mags')] to: kraken2-reports-mags.qza
Saved FeatureData[Kraken2Output % Properties('mags')] to: kraken2-hits-mags.qza


In [7]:
%%bash
qiime annotate kraken2-to-mag-features \
    --i-reports kraken2-reports-mags.qza \
    --i-outputs kraken2-hits-mags.qza \
    --o-taxonomy mags-taxonomy.qza \
    --verbose


/home/sadia/miniconda3/envs/rachis-moshpit-2026.4/lib/python3.12/site-packages/q2_annotate/kraken2/select.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  table = pd.DataFrame(rows).fillna(False)


Saved FeatureData[Taxonomy] to: mags-taxonomy.qza


STEP 5. MAG abundance estimation
i.MAG indexing
ii.Read mapping
iii. MAG length estimation
iv.Abundance estimation
v.Taxonomic composition visualization

In [8]:
%%bash
qiime assembly index-derep-mags \
    --i-mags mags-derep.qza \
    --p-threads 4 \
    --p-seed 100 \
    --o-index mags-derep-index.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: bowtie2-build --bmaxdivn 4 --dcv 1024 --offrate 5 --ftabchars 10 --threads 4 --seed 100 /tmp/tmpxbd1hoqo/merged.fasta /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index

Settings:
  Output files: "/tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 5 (one in 32)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 100
  Sizeofs: void*

Building a SMALL index
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.3.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.3.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.4.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.4.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.1.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.1.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.2.bt2.tmp to /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.2.bt2
Renaming /tmp/qiime2/sadia/processes/230948-1781095254.83@sadia/tmp/rachis-OutPath-j4q0_dds/index.rev.1.bt2.tmp to /tmp/qiime2/sadia/processes/230948-178

In [9]:
%%bash
qiime assembly map-reads \
    --i-index mags-derep-index.qza \
    --i-reads reads.qza \
    --p-threads 2 \
    --p-seed 100 \
    --o-alignment-maps reads-to-mags-aln.qza \
    --parallel-config parallel.config.toml \
    --verbose


Saved FeatureData[AlignmentMap] to: reads-to-mags-aln.qza


In [10]:
%%bash
qiime annotate get-feature-lengths \
    --i-features mags-derep.qza \
    --o-lengths mags-derep-lengths.qza \
    --verbose


Saved FeatureData[SequenceCharacteristics % Properties('length')] to: mags-derep-lengths.qza


In [11]:
%%bash
qiime annotate estimate-abundance \
    --i-alignment-maps reads-to-mags-aln.qza \
    --i-feature-lengths mags-derep-lengths.qza \
    --p-metric tpm \
    --p-min-mapq 42 \
    --p-threads 4 \
    --o-abundances mags-abundances.qza \
    --verbose


Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools view -q 42 -m 0 --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample1_alignment.bam | samtools sort -o /tmp/tmpunrzf5rx/sample1.bam --threads 4 /tmp/qiime2/sadia/data/b378b361-928b-45f8-9866-d8193dc9ade2/data/sample1_alignment.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: samtools coverage -Q 0 -l 0 -o /tmp/tmpunrzf5rx/sample1.coverage.tsv /tmp/tmpunrzf5rx/sample1.bam

Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These comman

[bam_sort_core] merging from 0 files and 4 in-memory blocks...
[bam_sort_core] merging from 0 files and 4 in-memory blocks...
[bam_sort_core] merging from 0 files and 4 in-memory blocks...
[bam_sort_core] merging from 0 files and 4 in-memory blocks...


In [12]:
%%bash
qiime taxa barplot \
    --i-table mags-abundances.qza \
    --i-taxonomy mags-taxonomy.qza \
    --o-visualization mags-taxa-barplot.qzv \
    --verbose


Saved Visualization to: mags-taxa-barplot.qzv
